In [ ]:
# Import functions from preprocessing.py
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# Add the path to the src directory
sys.path.append(os.path.abspath(os.path.join('..', 'data')))

In [ ]:
# Import functions from generation.py
from generation import generate_data, partition_domain_into_regions

In [ ]:
B = 64  # Number of regions
n_i = 30 # Number of locations per region

# Generate data
y, x, w, e, s, region_assignments = generate_data(B, n_i, sigmasq = 5, length_scale=1.0, nu=0.5, seed=50, beta_true=2, tau_true=1)

In [ ]:
# Plot the grid and points
grid_size = int(np.ceil(np.sqrt(B)))

# Create the plot
plt.figure(figsize=(8, 8))

# Plot the grid
x_divisions = np.linspace(0, 1, grid_size + 1)
y_divisions = np.linspace(0, 1, grid_size + 1)
for xx in x_divisions:
    plt.axvline(x=xx, color='k', linestyle='--', alpha=0.5)
for yy in y_divisions:
    plt.axhline(y=yy, color='k', linestyle='--', alpha=0.5)

# Plot the points
plt.scatter(s[:, 0], s[:, 1], c=region_assignments, cmap='tab10', marker='o', edgecolor='k', s=30)

# Title and labels
plt.title("Partitioned Grid with Points")
plt.xlabel("X-coordinate")
plt.ylabel("Y-coordinate")
plt.colorbar(label='Region ID')

plt.show()

In [ ]:
y.shape, x.shape, s.shape, region_assignments.shape, w.shape, e.shape

In [ ]:
# Optimizing using Numpy
import numpy as np
from scipy.optimize import minimize
from sklearn.gaussian_process.kernels import Matern
from scipy.spatial.distance import cdist

# Function to compute the NNGP covariance matrix using locations s
def compute_K_NN(s, K, num_neighbors):
    n = s.shape[0]
    K_NN = np.zeros_like(K)
    
    # Calculate pairwise distances between the locations s
    distances = cdist(s, s)
    
    for i in range(n):
        # Find the indices of the nearest neighbors based on location distances
        neighbor_indices = np.argsort(distances[i])[1:num_neighbors+1]
        K_NN[i, neighbor_indices] = K[i, neighbor_indices]
        K_NN[neighbor_indices, i] = K[neighbor_indices, i]
        
    return K_NN

def nngp_log_likelihood(y, s, X, beta, K, tausq, num_neighbors):
    n = len(y)
    K_NN = K # Use location s here
    K_NN_noise = K_NN 
    
    try:
        L = np.linalg.cholesky(K_NN_noise)
        log_det = 2 * np.sum(np.log(np.diagonal(L)))
    except np.linalg.LinAlgError:
        return -np.inf
    
    residual = y - X @ beta
    solve_term = np.linalg.solve(L.T, np.linalg.solve(L, residual))
    quadratic_term = residual.T @ solve_term
    
    log_likelihood = -0.5 * (n * np.log(2 * np.pi) + log_det + quadratic_term)
    return log_likelihood

# Function to compute the NNGP log-likelihood for given nu, length_scale, tausq, and beta
def nngp_log_likelihood_with_params(params, s, X, y, num_neighbors):
    # Extract parameters from the input
    n = len(y)
    nu = params[0]
    length_scale = params[1]
    tausq = params[2]
    sigmasq= params[3]
    beta = params[4:]
    
    # Compute the full covariance matrix K using the skMatern kernel
    matern_kernel = sigmasq * Matern(length_scale=length_scale, nu=0.5)
    K = matern_kernel(s) + tausq * np.eye(n) # Calculate the kernel matrix using location s
    
    # Compute the NNGP log-likelihood
    return -nngp_log_likelihood(y, s, X, beta, K, tausq, num_neighbors)  # Negative for minimization


# Initial guess for parameters: [nu, length_scale, tausq, beta_1, ..., beta_p]
params_init = np.array([0.5,0.3, 0.3, 2] + list(np.zeros(1)))
num_neighbors= 3

bounds = [(0.499, 0.501), (0.1, 10.0), (0.1, 10.0), (0.1, 10.0)] + [(None, None)] * (params_init.shape[0] - 4)

# Optimization with bounds
result = minimize(
    nngp_log_likelihood_with_params,
    params_init,
    args=(s, x, y, num_neighbors),
    method='L-BFGS-B',bounds=bounds
)

# Output the optimized parameters
print("Optimized Parameters:")
print("nu:", result.x[0])
print("length_scale:", result.x[1])
print("sigmasq/length_scale:", result.x[3]/result.x[1])
print("tausq:", result.x[2])
print("sigmasq:", result.x[3])
print("beta:", result.x[4:])
print("Log-Likelihood:", -result.fun)


In [ ]:
# Optimising using PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.gaussian_process.kernels import Matern
from scipy.spatial.distance import cdist
import numpy as np
import gpytorch
import torch
import torch.special

def exponential_kernel(X1, X2, length_scale=1.0, sigma=1.0):
    """
    Computes the Matérn covariance function with ν = 1/2 (exponential covariance).

    Args:
        X1 (torch.Tensor): Input tensor of shape (n, d).
        X2 (torch.Tensor): Input tensor of shape (m, d).
        rho (float): Characteristic lengthscale parameter.
        sigma (float): Scaling factor (variance term).

    Returns:
        torch.Tensor: Covariance matrix of shape (n, m).
    """
    dists = torch.cdist(X1, X2, p=2)  # Compute pairwise Euclidean distances
    return sigma * torch.exp(-dists / length_scale)

def compute_regionwise_covariance(s, region_assignments, sigmasq, length_scale, nu=0.5, tausq=1):
    """
    Computes the variance-covariance matrix for the averaged spatial GP process w_ibar.

    Parameters:
    s: (N, 2) tensor of spatial locations
    region_assignments: (N,) tensor assigning each location to a region
    sigma_sq: scalar, variance of the GP
    length_scale: scalar, length scale of the Matern kernel
    nu: scalar, smoothness parameter of the Matern kernel
    tausq: scalar, variance of the noise term

    Returns:
    cov_matrix: (B, B) tensor, variance-covariance matrix of region-averaged w
    """
    unique_regions = torch.unique(region_assignments)
    B = len(unique_regions)
    
    # Compute full covariance matrix for all locations using Matern kernel
    #scaled_s = s / length_scale
    matern_kernel = exponential_kernel(s, s, length_scale=length_scale)
    K = sigmasq * matern_kernel  # Scale by variance
    
    # Initialize variance-covariance matrix
    cov_matrix = torch.zeros((B, B))
    
    # Create a binary indicator matrix of shape (num_samples, num_regions)
    region_mask = torch.stack([(region_assignments == region).float() for region in unique_regions], dim=1)

    # Compute region sizes (n_i for each region)
    region_sizes = region_mask.sum(dim=0, keepdim=True)  # Shape: (1, num_regions)

    # Compute the covariance matrix efficiently
    # Equivalent to looping over i, j but using matrix multiplication
    region_sums = region_mask.T @ K @ region_mask  # Shape: (num_regions, num_regions)
    cov_matrix = region_sums / (region_sizes.T @ region_sizes)  # Element-wise division

    # Ensure numerical stability by avoiding division by zero
    cov_matrix[region_sizes.T @ region_sizes == 0] = 0
    cov_matrix

    # Add diagonal tausq / n_i term
    diag_tau_term = torch.diag(tausq / region_sizes)* torch.eye(B)  # Shape: (num_regions, num_regions)
    cov_matrix += diag_tau_term
        
    return cov_matrix


# PyTorch Module for optimizing NNGP parameters
class NNGPModel(nn.Module):
    def __init__(self, input_dim, device='cpu'):
        super(NNGPModel, self).__init__()
        self.device = device

        # Learnable parameters
        self.nu = nn.Parameter(torch.tensor(0.5, device=device))
        self.length_scale = nn.Parameter(torch.tensor(1.5, device=device))
        self.tausq = nn.Parameter(torch.tensor(1.0, device=device))
        self.sigmasq = nn.Parameter(torch.tensor(2.0, device=device))
        self.beta = nn.Parameter(torch.zeros(input_dim, device=device))
    
    def nngp_log_likelihood(self, y, s, X, beta, K):
        n = y.shape[0]
        K_NN = K  # Use precomputed covariance
        K_NN_noise = K_NN
        log_det = torch.logdet(K_NN_noise)
        
        residual = y - X @ beta
        quadratic_term = residual.T @ torch.linalg.solve(K_NN_noise, residual)
        log_likelihood = -0.5 * (log_det + quadratic_term)
        return log_likelihood   
        

    def forward(self, s, X, y):
        n = y.shape[0]
        # Compute covariance matrix using Matern kernel
        K = self.sigmasq * exponential_kernel(s, s, length_scale=self.length_scale) + self.tausq * torch.eye(n, device=self.device)
        return -self.nngp_log_likelihood(y, s, X, self.beta, K)

In [ ]:
# Sample Data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = 1
# num_neighbors= 3

y = torch.tensor(y, dtype=torch.float32, device=device)
x = torch.tensor(x, dtype=torch.float32, device=device)
w = torch.tensor(w, dtype=torch.float32, device=device)
e = torch.tensor(e, dtype=torch.float32, device=device)
s = torch.tensor(s, dtype=torch.float32, device=device)
region_assignments = torch.tensor(region_assignments, dtype=torch.int64, device=device)  # Region indices as integers


In [ ]:
# PyTorch Module for optimizing block-level covariance
class NNGPArealModel(nn.Module):
    def __init__(self, input_dim, device='cpu'):
        super(NNGPArealModel, self).__init__()
        self.device = device

        # Learnable parameters
        self.nu = nn.Parameter(torch.tensor(0.5, device=device))
        self.length_scale = nn.Parameter(torch.tensor(1.5, device=device))
        self.tausq = nn.Parameter(torch.tensor(1.0, device=device))
        self.sigmasq = nn.Parameter(torch.tensor(2.0, device=device))
        self.beta = nn.Parameter(torch.zeros(input_dim, device=device))
    
    def nngp_log_likelihood(self, y, s, X, beta, K):
        n = y.shape[0]
        K_NN = K  # Use precomputed covariance
        K_NN_noise = K_NN
        log_det = torch.logdet(K_NN_noise)
        
        residual = y - X @ beta
        quadratic_term = residual.T @ torch.linalg.solve(K_NN_noise, residual)
        log_likelihood = -0.5 * (log_det + quadratic_term)
        return log_likelihood   
        

    def forward(self, s, region_assignments, X, y):
        # Compute covariance matrix using Matern kernel
        K = compute_regionwise_covariance(s, region_assignments, self.sigmasq, self.length_scale, self.nu, self.tausq)
        return -self.nngp_log_likelihood(y, s, X, self.beta, K)

In [ ]:

# Initialize and optimize the model
model = NNGPModel(input_dim, device).to(device)
# optimizer = optim.LBFGS(model.parameters(), lr=0.1, max_iter=100, line_search_fn="strong_wolfe")
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(1000)):
    optimizer.zero_grad()
    loss = model(s, x, y)
    loss.backward()
    optimizer.step()
    
    # with torch.no_grad():
    #     model.length_scale.clamp_(min = .0001)
    #     model.tausq.clamp_(min = .0001)
    #     model.sigmasq.clamp_(min = .0001)
    #     model.nu.clamp_(min = .0001)
        
    if i % 100 == 0:
        print(f"Iteration {i}, Loss: {loss.item()}")
        print("nu:", model.nu.item())
        print("length_scale:", model.length_scale.item())
        print("sigma_sq/length_scale:", model.sigmasq.item()/(model.length_scale.item()))
        print("tausq:", model.tausq.item())
        print("sigmasq:", model.sigmasq.item())
        print("beta:", model.beta.detach().cpu().numpy())
        print("=="*20)

# # Optimization step
# def closure():
#     optimizer.zero_grad()
#     loss = model(s, x, y, num_neighbors)
#     loss.backward()
#     return loss

# optimizer.step(closure)
# with torch.no_grad():
#     print("Optimized Parameters:")
#     print("nu:", model.nu.item())
#     print("length_scale:", model.length_scale.item())
#     print("tausq:", model.tausq.item())
#     print("sigmasq:", model.sigmasq.item())
    

# # Output the optimized parameters
# print("Optimized Parameters:")
# print("nu:", model.nu.item())
# print("length_scale:", model.length_scale.item())
# print("tausq:", model.tausq.item())
# print("sigmasq:", model.sigmasq.item())
# print("beta:", model.beta.detach().cpu().numpy())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = 1
# num_neighbors= 3

y = torch.tensor(y, dtype=torch.float32, device=device)
x = torch.tensor(x, dtype=torch.float32, device=device)
w = torch.tensor(w, dtype=torch.float32, device=device)
e = torch.tensor(e, dtype=torch.float32, device=device)
s = torch.tensor(s, dtype=torch.float32, device=device)
region_assignments = torch.tensor(region_assignments, dtype=torch.int64, device=device)  # Region indices as integers



# Compute region-wise averages directly
unique_regions = torch.unique(region_assignments)
B = len(unique_regions)

ybar = torch.zeros(B, device=y.device)
xbar = torch.zeros(B, input_dim, device=x.device)

for i, region in enumerate(unique_regions):
    # Get indices for the current region
    indices = torch.where(region_assignments == region)[0]
    
    # Compute region-wise averages for y and x
    ybar[i] = torch.mean(y[indices])
    xbar[i] = torch.mean(x[indices], dim=0)

# Print results
print("Region-wise y averages (ybar):", ybar)
print("Region-wise x averages (xbar):", xbar)

# Initialize and optimize the model
model = NNGPArealModel(input_dim, device).to(device)
# optimizer = optim.LBFGS(model.parameters(), lr=0.1, max_iter=100, line_search_fn="strong_wolfe")
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(1000)):
    optimizer.zero_grad()
    loss = model(s, region_assignments, xbar, ybar)
    loss.backward()
    optimizer.step()
    
    #with torch.no_grad():
    #     model.length_scale.clamp_(min = .0001)
    #  model.tausq.clamp_(min = .0001)
    #     model.sigmasq.clamp_(min = .0001)
    #     model.nu.clamp_(min = .0001)
        
    if i % 100 == 0:
        print(f"Iteration {i}, Loss: {loss.item()}")
        print("nu:", model.nu.item())
        print("length_scale:", model.length_scale.item())
        print("sigma_sq/length_scale:", model.sigmasq.item()/(model.length_scale.item()))
        print("tausq:", model.tausq.item())
        print("sigmasq:", model.sigmasq.item())
        print("beta:", model.beta.detach().cpu().numpy())
        print("=="*20)
